<div dir="rtl" align="right">

# اختيارُ القنواتِ بِالحذفِ التكراريِّ للسماتِ (RFE)

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَستخدمُ RFE معَ SVM خطّيّ لاختيارِ أهمِّ سماتِ قدرَةِ النطاقات.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ لِترتيبِ السماتِ حسبَ رتبةِ RFE
- خريطةُ الرأسِ بِالقنواتِ المُختارة

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| N_SELECT | 10 |
| BANDS | alpha, beta |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>

In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تطبيقُ RFE

</div>

In [ ]:
from scipy.signal import welch
from sklearn.svm import SVC
from sklearn.feature_selection import RFE

FS = 250
N_SELECT = 10
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

estimator = SVC(kernel='linear')
selector = RFE(estimator, n_features_to_select=N_SELECT)
selector.fit(features, labels)
selected_mask = selector.support_
print(f'Selected {N_SELECT} features out of {len(selected_mask)}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- RFE يَأخذُ في الحسبانِ العلاقاتِ بينَ السمات
- قدْ يَختارُ قنواتٍ مُختلفةً عنْ SNR

</div>

In [ ]:
import plotly.graph_objects as go

rankings = selector.ranking_
sorted_idx = np.argsort(rankings)
colors = ['green' if selected_mask[i] else 'gray' for i in sorted_idx]
fig = go.Figure(go.Bar(x=[str(i) for i in sorted_idx], y=rankings[sorted_idx], marker_color=colors, name='RFE rank'))
fig.update_layout(height=500, title='RFE Feature Ranking (1=best)', xaxis_title='Feature index', yaxis_title='Rank')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- RFE يَحذفُ السماتِ تكراريّاً بِناءً على أهميّتِها في التصنيف
- يَكشفُ العلاقاتِ التفاعليّةَ بينَ السمات
- أكثرُ دقّةً من SNR لكنّهُ أبطأُ لِأنّهُ يُدرّبُ النموذجَ مِئاتِ المرّات

</div>